# 03A — Per-region analysis  ·  line A (NOAA)

One active region at a time, through the same eight steps every time. Every per-region knob
lives in `src/config.py` (`REGION_PARAMS`), so a block differs from its neighbours only by
which region it picks — not by retyped thresholds.

| step | what | where |
|---|---|---|
| 1 | load cubes, build regions | `loaders.load_noaa_region` |
| 2 | look at the masks | `analysis.plot_calibration_frame` / `plot_magnetogram_masks` / `plot_histogram` |
| 3 | per-frame metrics | `analysis.compute_metrics` / `save_metrics_csv` |
| 4 | time series | `analysis.plot_time_series` / `plot_area` |
| 5 | spectra | `spectra.plot_spectra_*`, `band_amplitude` |
| 6 | detrending | `oscillation.plot_moving_average` / `plot_filtered` |
| 7 | geometry check | `analysis.plot_mu_means` / `plot_mu_vs_trend` |
| 8 | animation | `animation.save_animation` |

`03B_data_analysis.ipynb` runs the identical eight steps on the DS0N datasets, so the two
notebooks can be read side by side.

**The regions are defined here, not in 02A.** `02A` writes only the three corrected cubes;
umbra, penumbra, hot spot and quiet sun are rebuilt from them on every run of this notebook,
so changing a threshold costs one `03A` run and never a re-run of `02A`.

**Cross-region conclusions are not here.** The comparison table, the 24 h fits, the
Wilson-depression scatter — all of that moved to `04_data_comparison.ipynb`, so that
analysing one region no longer means re-running the conclusions for all of them.

### Reading the numbers

- **Intensities** are in DN/s, limb-darkening corrected. Thresholds are *fractions of each
  frame's own quiet sun*, so mask areas do not drift as the region rotates.
- **Velocities** are absolute LOS m/s: positive is away from the observer. The umbra's own
  motion is `mean_dop_umb − mean_dop_quiet`; the absolute value still carries whatever the
  `v_SDO` correction left behind, which is itself diurnal and so sits right on top of the
  period being measured. Always look at both.
- **B** is the line-of-sight field, not `|B|`, and is not corrected for `cos θ`.
- **Gaps** are NaN and stay NaN. Spectra interpolate across them internally, which is a
  compromise, not a fix — check the gap count in step 1 before trusting a spectrum.

In [1]:
import pathlib
import sys

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np
from IPython.display import FileLink

from src import analysis, animation, config, oscillation, spectra
from src.loaders import load_noaa_region
from src.post_processing import *

regions = config.processed_regions()
print(f'{len(regions)} processed region(s):')
for i, region_dir in enumerate(regions):
    params = config.params_for(region_dir)
    extra = {k: v for k, v in params.items() if k not in config.NOAA_DEFAULTS}
    print(f'  [{i}] {region_dir.name:26s} {extra if extra else ""}')
if not regions:
    print('  none — run 02A_data_processing.ipynb first')

5 processed region(s):
  [0] NOAA_11106_2010-09-16      {'frame_idx': None}
  [1] NOAA_11117_2010-10-27      {'frame_idx': 400, 'mag_fill_slots': 0, 'crop_to_data': True}
  [2] NOAA_11363_2011-12-06      {'frame_idx': 200, 'frame_idx_masks': 411}
  [3] NOAA_11536_2012-07-31      {'custom_valid_region': array([[False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       ...,
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True],
       [False, False, False, ...,  True,  True,  True]], shape=(139, 484)), 'frame_idx': 0, 'frame_idx_masks': 411}
  [4] NOAA_13131_2022-10-29      {'frame_idx': 200, 'frame_idx_masks': 411}


## Knobs for this run

`SAVE` writes every figure into each region's `plots/`. `DOMINANT_PERIOD_MIN` is the period
the detrending step removes — 1440 min is 24 h, the signal this project is about.
`RUN_MU` gates step 7, which loads one map per frame and is therefore slow. `RAW_COMPARE`
gates step 7b, which loads each region a second time from `data/raw/`.

In [2]:
SAVE = True
DOMINANT_PERIOD_MIN = 1440.0
RUN_MU = False
RAW_COMPARE = False
REMOVE_POLINOMIAL = True

---

# NOAA 11106  ·  2010-09-16

### Step 1 — Load the cubes and build the regions. `params` comes from `config.REGION_PARAMS`.

In [3]:
REGION_INDEX = 0

region_dir = regions[REGION_INDEX]
raw_dir    = config.RAW_DIR / region_dir.name
plots_dir  = region_dir / 'plots'

params = config.params_for(region_dir)
view   = config.view_params(params)

data = load_noaa_region(region_dir, raw_dir=raw_dir, **config.loader_kwargs(params))

print(region_dir.name)
report = analysis.describe_region(data)

NOAA_11106_2010-09-16
  cube shape (n_t, ny, nx) : (480, 648, 648)
  time span                : 95.8 h (480 frames @ 720 s)
  segmentation             : umbra < 39,292, penumbra < 58,938
  NaN (missing) frames     : {'cont': 28, 'mag': 28, 'dop': 28}
      cont: frame 32 = 2010-09-16 06:36
      cont: frame 33 = 2010-09-16 06:48
      cont: frame 34 = 2010-09-16 07:00
      cont: frame 35 = 2010-09-16 07:12
      cont: frame 36 = 2010-09-16 07:24
      cont: ... and 23 more
      mag: frame 32 = 2010-09-16 06:36
      mag: frame 33 = 2010-09-16 06:48
      mag: frame 34 = 2010-09-16 07:00
      mag: frame 35 = 2010-09-16 07:12
      mag: frame 36 = 2010-09-16 07:24
      mag: ... and 23 more
      dop: frame 32 = 2010-09-16 06:36
      dop: frame 33 = 2010-09-16 06:48
      dop: frame 34 = 2010-09-16 07:00
      dop: frame 35 = 2010-09-16 07:12
      dop: frame 36 = 2010-09-16 07:24
      dop: ... and 23 more
  cont & mag together      : 452 of 452 continuum frames
  cluster           

In [5]:
data.keys()

dict_keys(['cube_cont', 'cube_mag', 'cube_dop', 'cube_mag_qsun', 'cube_dop_qsun', 'time_h', 'n_t', 'cadence_s', 'timestamps', 'present', 'mag_filled', 'c_mean', 'doppler_terms', 'umbra', 'penumbra', 'both', 'hot_spot', 'qsun', 'i_qs', 'cluster_info', 'raw_area_px', 'umbra_thresh', 'penumbra_thresh', 'cluster_mode', 'umbra_frac', 'penumbra_frac'])

In [4]:
def _broadcast(c: np.ndarray) -> np.ndarray:
    """(n_t,) -> (n_t, 1, 1) for broadcasting against (n_t, ny, nx) cubes."""
    return np.asarray(c, dtype=np.float32)[:, None, None]


def reconstruct_cubes(cube_dop, cube_mag, obs_times, k=K, pad_hours=24.0, verbose=True):
    """Rebuild dopplergram and magnetogram cubes with HMI's cubic calibration inverted.

    Generalised from the loop that used to sit in `01B`: fetch the coefficients covering
    the observation window, match each frame to the 12 h window it falls in, split V and B
    into the two circular polarisations, invert the cubic on each, and recombine.

    Returns `(cube_dop_reconstructed, cube_mag_reconstructed)`.
    """
    from astropy.time import Time

    times = Time(obs_times)
    coefficients = fetch_coefficients(times.min(), times.max(), pad_hours=pad_hours)
    matched = match_coefficients_to_times(times, coefficients)

    print(matched[0].size)

    c0, c1, c2, c3 = (_broadcast(matched[i]) for i in range(4))

    v_lcp, v_rcp = vlcp_rcp_from_v_b(cube_dop, cube_mag, k=k)
    v_lcp_raw = invert_cubic(v_lcp, c0, c1, c2, c3)
    v_rcp_raw = invert_cubic(v_rcp, c0, c1, c2, c3)
    dop, mag = v_b_from_vlcp_rcp(v_lcp_raw, v_rcp_raw, k=k)

    if verbose:
        print(f'{len(matched)} frame(s) matched to {coefficients["T_REC"].nunique()} '
              f'coefficient window(s)')
    return dop, mag

new_dop, new_mag = reconstruct_cubes(
    data['cube_dop'],
    data['cube_mag'],
    data['timestamps']
)

2026-08-17 00:32:14 - astropy - WARNING: TimeDeltaMissingUnitWarning: Numerical value without unit or explicit format passed to TimeDelta, assuming days


480


MemoryError: Unable to allocate 360. GiB for an array with shape (480, 1, 480, 648, 648) and data type float32

In [ ]:
if REMOVE_POLINOMIAL:
    reconstructed = reconstruct_cubes(data, view, remove_polynomial=True

### Step 2 — Look at what was actually selected before believing any number derived from it. `frame_idx` is a viewing choice from the registry and affects nothing.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

# The first and last frames are where a mask goes wrong without it showing anywhere else.
for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual. Written to `metrics.csv` so `04` can read it back without re-running this notebook.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, region_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

# Polarity balance: a mask that has drifted onto the wrong polarity, or straddles both
# poles of a bipolar group, shows up here and nowhere else.
for name in ('umbra', 'hot_spot'):
    if data.get(name) is None:
        continue
    values = np.where(data[name], data['cube_mag'], np.nan)
    finite = np.isfinite(values)
    if finite.any():
        positive = np.nansum(values > 0) / finite.sum()
        print(f'  {name:9s} polarity : {100 * positive:5.1f}% positive, '
              f'{100 * (1 - positive):5.1f}% negative')

### Step 4 — Raw, normalised, quiet-sun-subtracted, and magnetogram-residual views of the same series. The quiet-subtracted one is the umbra's own motion.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra. `region_spectra` builds all four regions at once; everything below is the same estimator asked different questions.

In [ ]:
cadence_s = data['cadence_s']

# Amplitude spectrum with peak table — "which periods are there, and how strong".
fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

# PSD — density-normalised, so regions of different length stay comparable.
psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

# Doppler against the magnetogram *residual*, not the raw magnetogram: the raw field's slow
# trend dominates its own spectrum, so comparing against it is the weaker test. If the two
# share a period, the field and the velocity are moving together.
psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

# The band-limited amplitude: one number per region, in m/s, for the ~24 h signal.
print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
# A window of one full period averages that period away — whatever is left is either a
# slower trend or something the 24 h signal was hiding.
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)

# The notch is blunt: it removes the tone *and* whatever real signal shares those bins, and
# zeroing a band rings in the time domain. Always read it next to the before/after spectrum.
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# mu = cos(heliocentric angle). If the magnetogram's slow trend tracks mu, the trend is the
# spot's changing foreshortening as it rotates, not something solar.
#
# This costs one map load per frame, so it is a step you run deliberately.
if RUN_MU:
    cube_mu = analysis.mu_cube_noaa(raw_dir, data['timestamps'], data['cadence_s'])
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
else:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')

### Step 7b — Optional — the same region straight from `data/raw/`, for comparison.

In [ ]:
# What did 02A's corrections actually remove? Loads the same region from data/raw/ and puts
# the two side by side.
#
# The raw dopplergram has the whole calibration unapplied — v_SDO alone is ~3 km/s and
# diurnal, sitting exactly on the period this project measures. So this figure is for
# seeing the correction, never for reading an amplitude off.
if RAW_COMPARE:
    data_raw = load_noaa_region(region_dir, raw_dir=raw_dir,
                                **config.loader_kwargs(params), **config.RAW_COMPARE)
    metrics_raw = analysis.compute_metrics(data_raw)
    print('raw (uncorrected) cubes:')
    analysis.plot_time_series(data_raw, metrics_raw, save=False, plots_dir=plots_dir)

    for suffix, label in (('umb', 'Umbra'), ('quiet', 'Quiet Sun')):
        raw_v = np.nanmean(metrics_raw[f'mean_dop_{suffix}'])
        cor_v = np.nanmean(metrics[f'mean_dop_{suffix}'])
        print(f'  {label:<10} mean v: raw {raw_v:9.1f} -> corrected {cor_v:9.1f} m/s '
              f'({raw_v - cor_v:+.1f} removed)')
else:
    print('RAW_COMPARE is False — set it in the setup cell to compare against data/raw/.')

### Step 8 — Animation, for looking at the region rather than at its averages.

In [ ]:
anim_path = animation.save_animation(data, metrics, region_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# NOAA 11117  ·  2010-10-27

### Step 1 — Load the cubes and build the regions. `params` comes from `config.REGION_PARAMS`.

In [ ]:
REGION_INDEX = 1

region_dir = regions[REGION_INDEX]
raw_dir    = config.RAW_DIR / region_dir.name
plots_dir  = region_dir / 'plots'

params = config.params_for(region_dir)
view   = config.view_params(params)

data = load_noaa_region(region_dir, raw_dir=raw_dir, **config.loader_kwargs(params))

print(region_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it. `frame_idx` is a viewing choice from the registry and affects nothing.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

# The first and last frames are where a mask goes wrong without it showing anywhere else.
for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual. Written to `metrics.csv` so `04` can read it back without re-running this notebook.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, region_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

# Polarity balance: a mask that has drifted onto the wrong polarity, or straddles both
# poles of a bipolar group, shows up here and nowhere else.
for name in ('umbra', 'hot_spot'):
    if data.get(name) is None:
        continue
    values = np.where(data[name], data['cube_mag'], np.nan)
    finite = np.isfinite(values)
    if finite.any():
        positive = np.nansum(values > 0) / finite.sum()
        print(f'  {name:9s} polarity : {100 * positive:5.1f}% positive, '
              f'{100 * (1 - positive):5.1f}% negative')

### Step 4 — Raw, normalised, quiet-sun-subtracted, and magnetogram-residual views of the same series. The quiet-subtracted one is the umbra's own motion.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra. `region_spectra` builds all four regions at once; everything below is the same estimator asked different questions.

In [ ]:
cadence_s = data['cadence_s']

# Amplitude spectrum with peak table — "which periods are there, and how strong".
fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

# PSD — density-normalised, so regions of different length stay comparable.
psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

# Doppler against the magnetogram *residual*, not the raw magnetogram: the raw field's slow
# trend dominates its own spectrum, so comparing against it is the weaker test. If the two
# share a period, the field and the velocity are moving together.
psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

# The band-limited amplitude: one number per region, in m/s, for the ~24 h signal.
print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
# A window of one full period averages that period away — whatever is left is either a
# slower trend or something the 24 h signal was hiding.
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)

# The notch is blunt: it removes the tone *and* whatever real signal shares those bins, and
# zeroing a band rings in the time domain. Always read it next to the before/after spectrum.
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# mu = cos(heliocentric angle). If the magnetogram's slow trend tracks mu, the trend is the
# spot's changing foreshortening as it rotates, not something solar.
#
# This costs one map load per frame, so it is a step you run deliberately.
if RUN_MU:
    cube_mu = analysis.mu_cube_noaa(raw_dir, data['timestamps'], data['cadence_s'])
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
else:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')

### Step 7b — Optional — the same region straight from `data/raw/`, for comparison.

In [ ]:
# What did 02A's corrections actually remove? Loads the same region from data/raw/ and puts
# the two side by side.
#
# The raw dopplergram has the whole calibration unapplied — v_SDO alone is ~3 km/s and
# diurnal, sitting exactly on the period this project measures. So this figure is for
# seeing the correction, never for reading an amplitude off.
if RAW_COMPARE:
    data_raw = load_noaa_region(region_dir, raw_dir=raw_dir,
                                **config.loader_kwargs(params), **config.RAW_COMPARE)
    metrics_raw = analysis.compute_metrics(data_raw)
    print('raw (uncorrected) cubes:')
    analysis.plot_time_series(data_raw, metrics_raw, save=False, plots_dir=plots_dir)

    for suffix, label in (('umb', 'Umbra'), ('quiet', 'Quiet Sun')):
        raw_v = np.nanmean(metrics_raw[f'mean_dop_{suffix}'])
        cor_v = np.nanmean(metrics[f'mean_dop_{suffix}'])
        print(f'  {label:<10} mean v: raw {raw_v:9.1f} -> corrected {cor_v:9.1f} m/s '
              f'({raw_v - cor_v:+.1f} removed)')
else:
    print('RAW_COMPARE is False — set it in the setup cell to compare against data/raw/.')

### Step 8 — Animation, for looking at the region rather than at its averages.

In [ ]:
anim_path = animation.save_animation(data, metrics, region_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# NOAA 11363  ·  2011-12-06

### Step 1 — Load the cubes and build the regions. `params` comes from `config.REGION_PARAMS`.

In [ ]:
REGION_INDEX = 2

region_dir = regions[REGION_INDEX]
raw_dir    = config.RAW_DIR / region_dir.name
plots_dir  = region_dir / 'plots'

params = config.params_for(region_dir)
view   = config.view_params(params)

data = load_noaa_region(region_dir, raw_dir=raw_dir, **config.loader_kwargs(params))

print(region_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it. `frame_idx` is a viewing choice from the registry and affects nothing.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

# The first and last frames are where a mask goes wrong without it showing anywhere else.
for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual. Written to `metrics.csv` so `04` can read it back without re-running this notebook.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, region_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

# Polarity balance: a mask that has drifted onto the wrong polarity, or straddles both
# poles of a bipolar group, shows up here and nowhere else.
for name in ('umbra', 'hot_spot'):
    if data.get(name) is None:
        continue
    values = np.where(data[name], data['cube_mag'], np.nan)
    finite = np.isfinite(values)
    if finite.any():
        positive = np.nansum(values > 0) / finite.sum()
        print(f'  {name:9s} polarity : {100 * positive:5.1f}% positive, '
              f'{100 * (1 - positive):5.1f}% negative')

### Step 4 — Raw, normalised, quiet-sun-subtracted, and magnetogram-residual views of the same series. The quiet-subtracted one is the umbra's own motion.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra. `region_spectra` builds all four regions at once; everything below is the same estimator asked different questions.

In [ ]:
cadence_s = data['cadence_s']

# Amplitude spectrum with peak table — "which periods are there, and how strong".
fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

# PSD — density-normalised, so regions of different length stay comparable.
psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

# Doppler against the magnetogram *residual*, not the raw magnetogram: the raw field's slow
# trend dominates its own spectrum, so comparing against it is the weaker test. If the two
# share a period, the field and the velocity are moving together.
psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

# The band-limited amplitude: one number per region, in m/s, for the ~24 h signal.
print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
# A window of one full period averages that period away — whatever is left is either a
# slower trend or something the 24 h signal was hiding.
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)

# The notch is blunt: it removes the tone *and* whatever real signal shares those bins, and
# zeroing a band rings in the time domain. Always read it next to the before/after spectrum.
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# mu = cos(heliocentric angle). If the magnetogram's slow trend tracks mu, the trend is the
# spot's changing foreshortening as it rotates, not something solar.
#
# This costs one map load per frame, so it is a step you run deliberately.
if RUN_MU:
    cube_mu = analysis.mu_cube_noaa(raw_dir, data['timestamps'], data['cadence_s'])
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
else:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')

### Step 7b — Optional — the same region straight from `data/raw/`, for comparison.

In [ ]:
# What did 02A's corrections actually remove? Loads the same region from data/raw/ and puts
# the two side by side.
#
# The raw dopplergram has the whole calibration unapplied — v_SDO alone is ~3 km/s and
# diurnal, sitting exactly on the period this project measures. So this figure is for
# seeing the correction, never for reading an amplitude off.
if RAW_COMPARE:
    data_raw = load_noaa_region(region_dir, raw_dir=raw_dir,
                                **config.loader_kwargs(params), **config.RAW_COMPARE)
    metrics_raw = analysis.compute_metrics(data_raw)
    print('raw (uncorrected) cubes:')
    analysis.plot_time_series(data_raw, metrics_raw, save=False, plots_dir=plots_dir)

    for suffix, label in (('umb', 'Umbra'), ('quiet', 'Quiet Sun')):
        raw_v = np.nanmean(metrics_raw[f'mean_dop_{suffix}'])
        cor_v = np.nanmean(metrics[f'mean_dop_{suffix}'])
        print(f'  {label:<10} mean v: raw {raw_v:9.1f} -> corrected {cor_v:9.1f} m/s '
              f'({raw_v - cor_v:+.1f} removed)')
else:
    print('RAW_COMPARE is False — set it in the setup cell to compare against data/raw/.')

### Step 8 — Animation, for looking at the region rather than at its averages.

In [ ]:
anim_path = animation.save_animation(data, metrics, region_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# NOAA 11536  ·  2012-07-31

### Step 1 — Load the cubes and build the regions. `params` comes from `config.REGION_PARAMS`.

In [ ]:
REGION_INDEX = 3

region_dir = regions[REGION_INDEX]
raw_dir    = config.RAW_DIR / region_dir.name
plots_dir  = region_dir / 'plots'

params = config.params_for(region_dir)
view   = config.view_params(params)

data = load_noaa_region(region_dir, raw_dir=raw_dir, **config.loader_kwargs(params))

print(region_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it. `frame_idx` is a viewing choice from the registry and affects nothing.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

# The first and last frames are where a mask goes wrong without it showing anywhere else.
for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual. Written to `metrics.csv` so `04` can read it back without re-running this notebook.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, region_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

# Polarity balance: a mask that has drifted onto the wrong polarity, or straddles both
# poles of a bipolar group, shows up here and nowhere else.
for name in ('umbra', 'hot_spot'):
    if data.get(name) is None:
        continue
    values = np.where(data[name], data['cube_mag'], np.nan)
    finite = np.isfinite(values)
    if finite.any():
        positive = np.nansum(values > 0) / finite.sum()
        print(f'  {name:9s} polarity : {100 * positive:5.1f}% positive, '
              f'{100 * (1 - positive):5.1f}% negative')

### Step 4 — Raw, normalised, quiet-sun-subtracted, and magnetogram-residual views of the same series. The quiet-subtracted one is the umbra's own motion.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra. `region_spectra` builds all four regions at once; everything below is the same estimator asked different questions.

In [ ]:
cadence_s = data['cadence_s']

# Amplitude spectrum with peak table — "which periods are there, and how strong".
fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

# PSD — density-normalised, so regions of different length stay comparable.
psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

# Doppler against the magnetogram *residual*, not the raw magnetogram: the raw field's slow
# trend dominates its own spectrum, so comparing against it is the weaker test. If the two
# share a period, the field and the velocity are moving together.
psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

# The band-limited amplitude: one number per region, in m/s, for the ~24 h signal.
print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
# A window of one full period averages that period away — whatever is left is either a
# slower trend or something the 24 h signal was hiding.
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)

# The notch is blunt: it removes the tone *and* whatever real signal shares those bins, and
# zeroing a band rings in the time domain. Always read it next to the before/after spectrum.
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# mu = cos(heliocentric angle). If the magnetogram's slow trend tracks mu, the trend is the
# spot's changing foreshortening as it rotates, not something solar.
#
# This costs one map load per frame, so it is a step you run deliberately.
if RUN_MU:
    cube_mu = analysis.mu_cube_noaa(raw_dir, data['timestamps'], data['cadence_s'])
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
else:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')

### Step 7b — Optional — the same region straight from `data/raw/`, for comparison.

In [ ]:
# What did 02A's corrections actually remove? Loads the same region from data/raw/ and puts
# the two side by side.
#
# The raw dopplergram has the whole calibration unapplied — v_SDO alone is ~3 km/s and
# diurnal, sitting exactly on the period this project measures. So this figure is for
# seeing the correction, never for reading an amplitude off.
if RAW_COMPARE:
    data_raw = load_noaa_region(region_dir, raw_dir=raw_dir,
                                **config.loader_kwargs(params), **config.RAW_COMPARE)
    metrics_raw = analysis.compute_metrics(data_raw)
    print('raw (uncorrected) cubes:')
    analysis.plot_time_series(data_raw, metrics_raw, save=False, plots_dir=plots_dir)

    for suffix, label in (('umb', 'Umbra'), ('quiet', 'Quiet Sun')):
        raw_v = np.nanmean(metrics_raw[f'mean_dop_{suffix}'])
        cor_v = np.nanmean(metrics[f'mean_dop_{suffix}'])
        print(f'  {label:<10} mean v: raw {raw_v:9.1f} -> corrected {cor_v:9.1f} m/s '
              f'({raw_v - cor_v:+.1f} removed)')
else:
    print('RAW_COMPARE is False — set it in the setup cell to compare against data/raw/.')

### Step 8 — Animation, for looking at the region rather than at its averages.

In [ ]:
anim_path = animation.save_animation(data, metrics, region_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

# NOAA 13131  ·  2022-10-29

### Step 1 — Load the cubes and build the regions. `params` comes from `config.REGION_PARAMS`.

In [ ]:
REGION_INDEX = 4

region_dir = regions[REGION_INDEX]
raw_dir    = config.RAW_DIR / region_dir.name
plots_dir  = region_dir / 'plots'

params = config.params_for(region_dir)
view   = config.view_params(params)

data = load_noaa_region(region_dir, raw_dir=raw_dir, **config.loader_kwargs(params))

print(region_dir.name)
report = analysis.describe_region(data)

### Step 2 — Look at what was actually selected before believing any number derived from it. `frame_idx` is a viewing choice from the registry and affects nothing.

In [ ]:
analysis.plot_calibration_frame(data, frame_idx=view['frame_idx'], save=SAVE, plots_dir=plots_dir)
analysis.plot_magnetogram_masks(data, frame_idx=view['frame_idx_masks'], save=SAVE, plots_dir=plots_dir)
analysis.plot_histogram(data, cube='continuum', save=SAVE, plots_dir=plots_dir)

# The first and last frames are where a mask goes wrong without it showing anywhere else.
for idx in (0, data['n_t'] - 1):
    analysis.plot_magnetogram_masks(data, frame_idx=idx, save=False, plots_dir=plots_dir)

### Step 3 — Per-frame means over each mask, plus the parabola-subtracted magnetogram residual. Written to `metrics.csv` so `04` can read it back without re-running this notebook.

In [ ]:
metrics = analysis.compute_metrics(data)
metrics = analysis.add_mag_residuals(data, metrics)
analysis.save_metrics_csv(data, metrics, region_dir)

umb, quiet = metrics['mean_dop_umb'], metrics['mean_dop_quiet']
print(f"  mean B umbra      : {np.nanmean(metrics['mean_mag_umb']):9.1f} G")
print(f"  mean v umbra      : {np.nanmean(umb):9.1f} m/s")
print(f"  mean v quiet sun  : {np.nanmean(quiet):9.1f} m/s")
print(f"  umbra - quiet     : {np.nanmean(umb - quiet):9.1f} m/s")

# Polarity balance: a mask that has drifted onto the wrong polarity, or straddles both
# poles of a bipolar group, shows up here and nowhere else.
for name in ('umbra', 'hot_spot'):
    if data.get(name) is None:
        continue
    values = np.where(data[name], data['cube_mag'], np.nan)
    finite = np.isfinite(values)
    if finite.any():
        positive = np.nansum(values > 0) / finite.sum()
        print(f'  {name:9s} polarity : {100 * positive:5.1f}% positive, '
              f'{100 * (1 - positive):5.1f}% negative')

### Step 4 — Raw, normalised, quiet-sun-subtracted, and magnetogram-residual views of the same series. The quiet-subtracted one is the umbra's own motion.

In [ ]:
analysis.plot_time_series(data, metrics, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, normalized=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, subtract_quiet=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_time_series(data, metrics, mag_residual=True, save=SAVE, plots_dir=plots_dir)
analysis.plot_area(data, metrics, save=SAVE, plots_dir=plots_dir)

### Step 5 — Spectra. `region_spectra` builds all four regions at once; everything below is the same estimator asked different questions.

In [ ]:
cadence_s = data['cadence_s']

# Amplitude spectrum with peak table — "which periods are there, and how strong".
fft = spectra.region_spectra(metrics, cadence_s, kind='fft')
spectra.plot_spectra_separate(fft, cadence_s, kind='fft', save=SAVE, plots_dir=plots_dir)
spectra.plot_spectra_combined(fft, cadence_s, kind='fft', xlim=(0, 0.1),
                              save=SAVE, plots_dir=plots_dir)

# PSD — density-normalised, so regions of different length stay comparable.
psd = spectra.region_spectra(metrics, cadence_s, kind='psd')
spectra.plot_spectra_combined(psd, cadence_s, kind='psd', xlim=(0, 0.1), log_y=True,
                              save=SAVE, plots_dir=plots_dir)

# Doppler against the magnetogram *residual*, not the raw magnetogram: the raw field's slow
# trend dominates its own spectrum, so comparing against it is the weaker test. If the two
# share a period, the field and the velocity are moving together.
psd_mag = spectra.region_spectra(metrics, cadence_s, kind='psd', key='mean_mag_residual')
spectra.plot_spectra_compare(psd, psd_mag, cadence_s, label_a='Dopplergram',
                             label_b='Magnetogram residual', kind='psd', normalize=True,
                             xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

# The band-limited amplitude: one number per region, in m/s, for the ~24 h signal.
print(f'{"region":<12} {"amplitude":>10} {"period":>9} {"cycles":>8}')
for label, suffix, _ in spectra.DEFAULT_REGIONS:
    series = metrics.get(f'mean_dop_{suffix}')
    if series is None:
        continue
    result = spectra.band_amplitude(data['time_h'], series)
    print(f'{label:<12} {result["amplitude"]:9.1f}  {result["period_h"]:8.2f} h '
          f'{result["n_cycles"]:8.1f}')

### Step 6 — Detrending — what is left once the dominant period is taken out.

In [ ]:
# A window of one full period averages that period away — whatever is left is either a
# slower trend or something the 24 h signal was hiding.
oscillation.plot_moving_average(data, metrics, window_min=DOMINANT_PERIOD_MIN,
                                save=SAVE, plots_dir=plots_dir)

# The notch is blunt: it removes the tone *and* whatever real signal shares those bins, and
# zeroing a band rings in the time domain. Always read it next to the before/after spectrum.
oscillation.plot_filtered(data, metrics, period_min=DOMINANT_PERIOD_MIN,
                          save=SAVE, plots_dir=plots_dir)
spectra.plot_spectrum_before_after(metrics['mean_dop_umb'], data['cadence_s'],
                                   period_min=DOMINANT_PERIOD_MIN, label='umbra',
                                   xlim=(0, 0.1), save=SAVE, plots_dir=plots_dir)

### Step 7 — Is the slow magnetogram trend just geometry?

In [ ]:
# mu = cos(heliocentric angle). If the magnetogram's slow trend tracks mu, the trend is the
# spot's changing foreshortening as it rotates, not something solar.
#
# This costs one map load per frame, so it is a step you run deliberately.
if RUN_MU:
    cube_mu = analysis.mu_cube_noaa(raw_dir, data['timestamps'], data['cadence_s'])
    mu_means, mu_fits = analysis.plot_mu_means(cube_mu, data, save=SAVE, plots_dir=plots_dir)
    analysis.plot_mu_vs_trend(data, metrics, mu_means, key='mean_mag',
                              save=SAVE, plots_dir=plots_dir)
else:
    print('RUN_MU is False — set it in the setup cell to run the geometry check.')

### Step 7b — Optional — the same region straight from `data/raw/`, for comparison.

In [ ]:
# What did 02A's corrections actually remove? Loads the same region from data/raw/ and puts
# the two side by side.
#
# The raw dopplergram has the whole calibration unapplied — v_SDO alone is ~3 km/s and
# diurnal, sitting exactly on the period this project measures. So this figure is for
# seeing the correction, never for reading an amplitude off.
if RAW_COMPARE:
    data_raw = load_noaa_region(region_dir, raw_dir=raw_dir,
                                **config.loader_kwargs(params), **config.RAW_COMPARE)
    metrics_raw = analysis.compute_metrics(data_raw)
    print('raw (uncorrected) cubes:')
    analysis.plot_time_series(data_raw, metrics_raw, save=False, plots_dir=plots_dir)

    for suffix, label in (('umb', 'Umbra'), ('quiet', 'Quiet Sun')):
        raw_v = np.nanmean(metrics_raw[f'mean_dop_{suffix}'])
        cor_v = np.nanmean(metrics[f'mean_dop_{suffix}'])
        print(f'  {label:<10} mean v: raw {raw_v:9.1f} -> corrected {cor_v:9.1f} m/s '
              f'({raw_v - cor_v:+.1f} removed)')
else:
    print('RAW_COMPARE is False — set it in the setup cell to compare against data/raw/.')

### Step 8 — Animation, for looking at the region rather than at its averages.

In [ ]:
anim_path = animation.save_animation(data, metrics, region_dir / 'anim.html',
                                     step=max(1, data['n_t'] // 60), fps=5,
                                     mag_symmetric_cbar=True)
FileLink(str(pathlib.Path(anim_path).relative_to(project_root)))

---

## Export the masks for DS9

A `uint8` bitmask cube per region — bit 0 umbra, bit 1 penumbra, bit 2 hot spot — carrying
the same spatial WCS as the cubes, so DS9 puts it on the right sky.

In [ ]:
for region_dir in regions:
    params = config.params_for(region_dir)
    d = load_noaa_region(region_dir, raw_dir=config.RAW_DIR / region_dir.name,
                         **config.loader_kwargs(params))
    from src.segmentation import write_masks_cube
    write_masks_cube(d, region_dir / 'region_01_masks_cube.fits',
                     history=[f'03A: masks rebuilt from the corrected cubes, '
                              f'params from config.REGION_PARAMS'])

---

## Next

`04_data_comparison.ipynb` takes the `metrics.csv` files written in step 3 and does
everything that compares regions to each other: the summary table, the 24 h fits, and the
amplitude-vs-Wilson-depression relation.